In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
df = pd.read_json('../data/Prepared/car_data.json')

## Data contract (Data Dictionary + Validity Rules)

The dataset in `data/raw/car_data.json` and `data/Prepared/car_data.json` currently uses the same column schema. The table below documents the expected meaning, type, and validation rules for each field.

So this works as checklist when cleaning data.

| Column | Meaning | Type | Valid range/values | Units | Notes |
| --- | --- | --- | --- | --- | --- |
| `url` | listing URL | string | expected to be a valid OLX posting URL | - | should identify a posting, but duplicates are present in the current data |
| `posting_date` | date the listing was posted | date string | usually `DD.MM.YYYY`; some rows use relative text such as `Сегодня` | date | must be normalized before date parsing |
| `region` | seller region in Uzbekistan | category | 13 observed region values including `Tashkent`, `Samarkand`, `Bukhara`, `Karakalpakstan` | - | 1 missing value observed |
| `district` | district or city within a region | category | 230 observed values | - | spelling and naming normalization may be needed |
| `price` | listed sale price | numeric stored as string | positive values; observed range `10,000,000` to `2,008,972,671` | local currency | 5 missing values observed; outliers should be reviewed |
| `currency` | currency code for `price` | category | mostly `UZS` when present | ISO currency code | highly sparse in the current data; many rows are missing |
| `description` | free-text ad description | text | arbitrary seller-written text | - | noisy multilingual text; useful for NLP/features after cleaning |
| `image_url` | main image link for the ad | string | expected `http`/`https` image URL | - | optional; missing values are present |
| `seller_type` | seller classification | category | mostly `private` in observed data | - | many missing values; could later include dealer/business labels |
| `model` | vehicle model name | category/text | 720 observed values | - | may mix trims, generations, and inconsistent naming |
| `body_type` | vehicle body style | category | `Sedan`, `Hatchback`, `Station Wagon`, `SUV`, `Minivan`, `Pickup`, `Coupe`, `Convertible`, `Other` | - | appears complete in the current snapshot |
| `sale_type` | sale terms attached to the listing | multi-label text/category | values such as `Simple Sale`, `Credit`, `Installment`, `Exchange`, `Rent`, often comma-separated combinations | - | multilingual and highly inconsistent; many missing values |
| `year` | vehicle manufacture year | integer stored as string | observed range `1900` to `2025` | year | should be checked against realistic car production years |
| `mileage` | vehicle mileage | integer stored as string | observed range `0` to `10,000,000` | km | missing values and obvious outliers are possible |
| `transmission` | gearbox type | category | `Manual`, `Automatic`, `Other` | - | appears complete in the current snapshot |
| `color` | vehicle color | category | 23 observed color values such as `White`, `Black`, `Gray`, `Blue` | - | synonyms may need consolidation |
| `engine_volume` | engine size | numeric stored as string | observed range `1` to `10000` | unclear/mixed | scale is inconsistent across rows and likely needs normalization before analysis |
| `fuel_type` | fuel system | category | `Gasoline/Gas`, `Gasoline`, `Electric`, `Hybrid`, `Diesel`, `Other` | - | appears complete in the current snapshot |
| `condition` | vehicle condition | category | `Excellent`, `Good`, `Average`, `Needs Repair` | - | ordinal meaning should be preserved if encoded |
| `owners_count` | number of previous owners | ordinal category stored as string | `1`, `2`, `3`, `4+` | owners | missing values are present; `4+` is grouped rather than exact |
| `additional_options` | extra features/options listed in the ad | comma-separated text list | examples include `Customs Cleared`, `Electrical Window Lifters`, parking sensors, etc. | - | multi-valued field; multilingual, sparse, and suitable for splitting into tags |

## Data intake & first checks (done for EACH file)

In [13]:
df.columns

Index(['url', 'posting_date', 'region', 'district', 'price', 'currency',
       'description', 'image_url', 'seller_type', 'model', 'body_type',
       'sale_type', 'year', 'mileage', 'transmission', 'color',
       'engine_volume', 'fuel_type', 'condition', 'owners_count',
       'additional_options'],
      dtype='object')

In [10]:
df.tail(5)

,url,posting_date,region,district,price,currency,description,image_url,seller_type,model,...,sale_type,year,mileage,transmission,color,engine_volume,fuel_type,condition,owners_count,additional_options
47101,https://olx.uz/d/obyavlenie/kia-bongo-2023-sot...,15.04.2025,Bukhara,Buhara,300000000,UZS,Propan òrnatilgan\nFurgon qilingan\nKandisione...,https://frankfurt.apollo.olxcdn.com:443/v1/fil...,private,: Другая,...,None,2023,5500.0,Manual,White,80,Gasoline/Gas,Excellent,4+,"Парктроник, Electrical Mirrors, Customs Cleare..."
47102,https://olx.uz/d/obyavlenie/inavate-elektro-mo...,13.05.2025,Surkhandarya,Termez,300000000,UZS,Mashina ideal oq rangi qiziriq tumanida inavat...,https://frankfurt.apollo.olxcdn.com:443/v1/fil...,private,: Другая,...,None,2021,30000.0,Automatic,White,3,Electric,Excellent,1,None
47103,https://olx.uz/d/obyavlenie/malibu-2-turbo-pri...,13.05.2025,Khorezm,Karaul,300000000,UZS,Йили 2019 пробег 81800 янги краска йук нархи 2...,https://frankfurt.apollo.olxcdn.com:443/v1/fil...,private,: Malibu,...,Simple Sale,2019,81800.0,Automatic,Black,2,Gasoline/Gas,Excellent,2,Air Conditioner
47104,https://olx.uz/d/obyavlenie/treaker-2-premier-...,10.05.2025,Navoi,Navoi,300000000,UZS,Mashina. Yangi. Salondan. Chiqanganiga e...,https://frankfurt.apollo.olxcdn.com:443/v1/fil...,private,: Tracker,...,None,2024,20.0,Automatic,Black,12,Gasoline,Excellent,4+,"Air Conditioner, Electrical Mirrors, Парктрони..."
47105,https://olx.uz/d/obyavlenie/malibu-2-alo-xolat...,08.05.2025,Surkhandarya,Sariasiya,280000000,UZS,"Kraska toza \n4 ta yumshoq balon\nMator, xadav...",https://frankfurt.apollo.olxcdn.com:443/v1/fil...,private,: Malibu,...,Simple Sale,2020,67000.0,Automatic,Black,15,Gasoline,Excellent,1,"Парктроник, Electrical Mirrors, Security Syste..."


##  Data quality audit (prove what’s wrong before fixing)

Data quality audit

|Column|% missing |Is missing acceptable?|Why might it be missing?|
|---|---|---|---|
|currency|73%|No|Scraper could not identify USD currency|
|sale_type|60%|Maybe|Sales type not included in description|
|model|1%|No|Model not included in description|
|mileage|7%|Yes(moderate, manageable)|Seller may be hide high usage, older listings may not include it|
|additional_options|34%|Yes|User forgot to include or none exist|

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 47106 entries, 0 to 47105
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   url                 47106 non-null  object 
 1   posting_date        47106 non-null  object 
 2   region              47105 non-null  object 
 3   district            47105 non-null  object 
 4   price               47101 non-null  object 
 5   currency            12571 non-null  object 
 6   description         47106 non-null  object 
 7   image_url           45520 non-null  object 
 8   seller_type         39798 non-null  object 
 9   model               47047 non-null  object 
 10  body_type           47106 non-null  object 
 11  sale_type           18713 non-null  object 
 12  year                47106 non-null  int64  
 13  mileage             43818 non-null  float64
 14  transmission        47106 non-null  object 
 15  color               47106 non-null  object 
 16  engine_vo

## Cleaning strategy (not just steps-decisions)